# Data Cleaning -- StudentLife Dataset
**MSc Data Science and AI | Visual Analytics for Digital Health**  
**Rohith Elanchezhian | Newcastle University | Supervisor: Alaa Alahmadi**

---

## Overview

This notebook handles the first stage of the project pipeline: loading and cleaning the raw StudentLife dataset.

The StudentLife dataset was collected at Dartmouth College in Spring 2013. It contains data from 49 undergraduate students over a 10-week period, captured through smartphone sensors and daily self-report surveys (called EMA -- Ecological Momentary Assessment).

The raw data comes in a mix of JSON files (EMA surveys) and CSV files (passive sensors). Before any analysis can happen, I need to:

1. Read all raw files from Google Drive
2. Convert Unix timestamps into proper readable dates in EDT (the students' local timezone)
3. Filter out readings outside the actual 10-week study period
4. Fix known data entry bugs in the survey files
5. Remove duplicates and out-of-range values
6. Save 9 clean CSV files ready for the EDA and ML stages

---

**Study period:** 27 March 2013 to 5 June 2013 (10 weeks)  
**Timezone:** All timestamps are stored as UTC in the raw data. Since the students were at Dartmouth College in New Hampshire, USA, I convert everything to EDT (Eastern Daylight Time = UTC-4) so that morning and evening patterns make sense.

## Step 1 -- Mount Google Drive

The dataset files are stored on Google Drive. This cell mounts the drive so the notebook can access them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted successfully.')

## Step 2 -- Import Libraries

I am using standard Python data science libraries throughout:

- **pandas** for loading and manipulating tabular data
- **numpy** for numerical operations and handling missing values
- **json** for reading the EMA survey files (stored as JSON)
- **os** for navigating the file system on Drive
- **matplotlib** for quick sanity-check plots at the end


In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Show more columns when printing DataFrames -- useful when checking data
pd.set_option('display.max_columns', 20)

print('All libraries imported successfully.')

## Step 3 -- Set the Data Directory Path

Change `DATA_DIR` below to point to wherever you saved the extracted StudentLife dataset on your Google Drive.

The folder should contain two sub-folders:
- `EMA/` -- the self-report survey responses (JSON files)
- `sensing/` -- the passive sensor readings (CSV files)


In [ ]:
# Change this path to match where you saved the dataset on your Drive
DATA_DIR = '/content/drive/MyDrive/dataset'

# Verify the folder structure looks right before going further
ema_path     = os.path.join(DATA_DIR, 'EMA')
sensing_path = os.path.join(DATA_DIR, 'sensing')

if os.path.exists(ema_path) and os.path.exists(sensing_path):
    print('Dataset folder found. Contents:')
    print()
    print('EMA survey folders:')
    for folder in sorted(os.listdir(os.path.join(ema_path, 'response'))):
        print(f'   {folder}')
    print()
    print('Sensing data types:')
    for folder in sorted(os.listdir(sensing_path)):
        print(f'   {folder}')
else:
    print('ERROR: Could not find the expected folder structure.')
    print(f'Looking inside: {DATA_DIR}')
    print('Please make sure DATA_DIR points to the correct folder.')

## Step 4 -- Constants and Helper Functions

I define the study period boundaries and a set of helper functions here that I reuse throughout.

**Important timezone note:** The raw data stores all timestamps as Unix timestamps (seconds since 1 January 1970) in UTC. The students were in Hanover, New Hampshire -- Eastern Daylight Time (EDT = UTC-4). If I do not convert to EDT, every time-based pattern in the analysis will be shifted by 4 hours, making it look like students had their most stressful moments at 3am when it was actually 7am. This is a critical cleaning step.

I also define a function that works out which study week (1-10) each reading belongs to, based on how many days it is after the study start date.

In [ ]:
# Study period -- these are the exact dates the study ran
# Source: Wang et al. (2014) StudentLife paper
STUDY_START = pd.Timestamp('2013-03-27', tz='America/New_York')
STUDY_END   = pd.Timestamp('2013-06-05', tz='America/New_York')

# All 49 student IDs follow the format u00, u01, ..., u48
ALL_STUDENTS = [f'u{i:02d}' for i in range(49)]


def convert_to_edt(timestamp_series):
    '''
    Convert a column of raw Unix timestamps into proper datetime objects in EDT.

    The raw data stores all times as seconds since 1 Jan 1970 (Unix epoch), in UTC.
    This function converts them to EDT (America/New_York) -- the timezone where
    the Dartmouth students actually were during the study.
    '''
    as_numbers = pd.to_numeric(timestamp_series, errors='coerce')
    as_utc     = pd.to_datetime(as_numbers, unit='s', utc=True)
    as_edt     = as_utc.dt.tz_convert('America/New_York')
    return as_edt


def within_study_period(datetime_column):
    '''
    Returns a boolean mask -- True for rows inside the 10-week study window.
    Anything before 27 March or after 5 June 2013 is removed.
    '''
    return (datetime_column >= STUDY_START) & (datetime_column <= STUDY_END)


def get_study_week(datetime_column):
    '''
    Maps each date to study week 1 through 10.
    Week 1 starts 27 March 2013. Week 10 (finals) ends 5 June 2013.
    '''
    days_since_start = (datetime_column - STUDY_START).dt.days
    week_number      = (days_since_start // 7) + 1
    return week_number.clip(1, 10)  # keep within valid range


def read_ema_json(survey_type, student_id):
    '''
    Load one student EMA survey responses from their JSON file.

    Args:
        survey_type -- the survey folder name, e.g. Stress, Sleep, Mood
        student_id  -- the student ID, e.g. u00, u01

    Returns:
        A list of response dictionaries, or empty list if file not found.
    '''
    file_path = os.path.join(
        DATA_DIR, 'EMA', 'response',
        survey_type, f'{survey_type}_{student_id}.json'
    )
    if not os.path.exists(file_path):
        return []
    with open(file_path, 'r') as f:
        return json.load(f)


def read_sensing_csv(sensor_type, student_id):
    '''
    Load one student passive sensor CSV file from Drive.

    Different sensor types use different file name prefixes -- this function
    handles that mapping automatically.

    Args:
        sensor_type -- e.g. bluetooth, conversation, activity
        student_id  -- e.g. u00, u01

    Returns:
        A pandas DataFrame, or None if the file does not exist.
    '''
    prefix_map = {
        'bluetooth':    'bt',
        'conversation': 'conversation',
        'activity':     'activity',
        'wifi_location':'wifi_location',
        'phonelock':    'phonelock',
    }
    prefix    = prefix_map.get(sensor_type, sensor_type)
    file_path = os.path.join(
        DATA_DIR, 'sensing', sensor_type,
        f'{prefix}_{student_id}.csv'
    )
    if not os.path.exists(file_path):
        return None
    return pd.read_csv(file_path, skipinitialspace=True)


print('Constants and helper functions defined.')
print(f'Study period: {STUDY_START.date()} to {STUDY_END.date()}')
print(f'Students:     {len(ALL_STUDENTS)} (u00 to u48)')

## Step 5 -- Mood EMA

The mood survey was split across three separate folders: `Mood`, `Mood 1`, and `Mood 2`. This happened because the study app was updated partway through the term. I collect responses from all three folders and merge them.

**Survey questions:**
- `happy` (1-5): How happy do you feel right now?
- `sad` (1-5): How sad do you feel right now?
- `tomorrow` (1-5): How do you feel about tomorrow?
- `how` (1-5): How are you feeling overall?

I compute a composite **mood score = happy minus sad**, ranging from -4 (very sad) to +4 (very happy). A score of zero means the student felt equally happy and sad.

In [ ]:
# Collect all mood responses across the three survey folders
all_mood = []

for student_id in ALL_STUDENTS:
    for folder in ['Mood', 'Mood 1', 'Mood 2']:
        responses = read_ema_json(folder, student_id)
        for r in responses:
            all_mood.append({
                'student_id': student_id,
                'folder':     folder,
                'resp_time':  r.get('resp_time'),
                'happy':      r.get('happy'),
                'sad':        r.get('sad'),
                'tomorrow':   r.get('tomorrow'),
                'how':        r.get('how'),
            })

mood_raw = pd.DataFrame(all_mood)
print(f'Raw mood responses collected: {len(mood_raw):,}')

In [ ]:
# Convert timestamps to proper EDT dates
mood_raw['datetime'] = convert_to_edt(mood_raw['resp_time'])

# Remove anything outside the study period
mood_raw = mood_raw[within_study_period(mood_raw['datetime'])].copy()
print(f'After filtering to study period: {len(mood_raw):,} rows')

# Convert rating columns to numbers and remove any out-of-range values
# All ratings should be 1-5; anything outside that is a data entry error
for col in ['happy', 'sad', 'tomorrow', 'how']:
    mood_raw[col] = pd.to_numeric(mood_raw[col], errors='coerce')
    out_of_range  = ~mood_raw[col].between(1, 5) & mood_raw[col].notna()
    mood_raw.loc[out_of_range, col] = np.nan

# Drop exact duplicates (same student, same timestamp, same folder)
before = len(mood_raw)
mood_raw.drop_duplicates(subset=['student_id', 'resp_time', 'folder'],
                         keep='first', inplace=True)
print(f'Removed {before - len(mood_raw)} duplicate rows')

# Calculate composite mood score and add date/time columns
mood_raw['mood_score'] = mood_raw['happy'] - mood_raw['sad']
mood_raw['date']        = mood_raw['datetime'].dt.date
mood_raw['hour']        = mood_raw['datetime'].dt.hour
mood_raw['day_of_week'] = mood_raw['datetime'].dt.dayofweek  # 0=Mon, 6=Sun
mood_raw['is_weekend']  = mood_raw['day_of_week'] >= 5
mood_raw['study_week']  = get_study_week(mood_raw['datetime'])

# Select the columns to keep
mood_clean = mood_raw[[
    'student_id', 'folder', 'datetime', 'date', 'hour',
    'day_of_week', 'is_weekend', 'study_week',
    'happy', 'sad', 'mood_score', 'tomorrow', 'how'
]].copy()

mood_clean.sort_values(['student_id', 'datetime'], inplace=True)
mood_clean.reset_index(drop=True, inplace=True)

print(f'\nFinal mood dataset: {len(mood_clean):,} rows from {mood_clean["student_id"].nunique()} students')
print(f'Mood score range: {mood_clean["mood_score"].min():.0f} to {mood_clean["mood_score"].max():.0f}')
print(f'Mean mood score:  {mood_clean["mood_score"].mean():.2f}')
mood_clean.head(3)

## Step 6 -- Stress EMA

The stress survey asks students to rate their current stress level from 1 (not stressed at all) to 5 (very stressed). This is the most important variable in the project -- it is the main target for the machine learning models.

The field in the raw JSON is `level`.

In [ ]:
all_stress = []

for student_id in ALL_STUDENTS:
    for r in read_ema_json('Stress', student_id):
        all_stress.append({
            'student_id':   student_id,
            'resp_time':    r.get('resp_time'),
            'stress_level': r.get('level')
        })

stress_raw = pd.DataFrame(all_stress)
print(f'Raw stress responses: {len(stress_raw):,}')

In [ ]:
# Convert timestamps and filter to study period
stress_raw['datetime'] = convert_to_edt(stress_raw['resp_time'])
stress_raw = stress_raw[within_study_period(stress_raw['datetime'])].copy()

# Convert to number and validate 1-5 range
stress_raw['stress_level'] = pd.to_numeric(stress_raw['stress_level'], errors='coerce')
out_of_range = ~stress_raw['stress_level'].between(1, 5) & stress_raw['stress_level'].notna()
stress_raw.loc[out_of_range, 'stress_level'] = np.nan

# Remove duplicates
before = len(stress_raw)
stress_raw.drop_duplicates(subset=['student_id', 'resp_time'], keep='first', inplace=True)
print(f'Removed {before - len(stress_raw)} duplicates')

# Add time columns
stress_raw['date']        = stress_raw['datetime'].dt.date
stress_raw['hour']        = stress_raw['datetime'].dt.hour
stress_raw['day_of_week'] = stress_raw['datetime'].dt.dayofweek
stress_raw['is_weekend']  = stress_raw['day_of_week'] >= 5
stress_raw['study_week']  = get_study_week(stress_raw['datetime'])

stress_clean = stress_raw[[
    'student_id', 'datetime', 'date', 'hour',
    'day_of_week', 'is_weekend', 'study_week', 'stress_level'
]].copy()

stress_clean.sort_values(['student_id', 'datetime'], inplace=True)
stress_clean.reset_index(drop=True, inplace=True)

print(f'\nFinal stress dataset: {len(stress_clean):,} rows from {stress_clean["student_id"].nunique()} students')
print(f'Mean stress level: {stress_clean["stress_level"].mean():.2f} / 5')
print(f'Distribution:')
print(stress_clean['stress_level'].value_counts().sort_index())

## Step 7 -- Sleep EMA

The sleep survey captures how many hours a student slept the previous night and their sleep quality.

**Fields:**
- `hour` -- hours of sleep (expected range: 0-16)
- `rate` -- sleep quality rating (1 = very bad, 5 = very good)

**Known data bug:** In some Sleep survey responses, the hours value is stored under the key `null` instead of `hour`. This appears to be a bug in the original data collection app. I check both keys and use whichever has a valid value.


In [ ]:
all_sleep = []

for student_id in ALL_STUDENTS:
    for r in read_ema_json('Sleep', student_id):

        # BUG FIX: Some entries store sleep hours under 'null' key instead of 'hour'
        # This was a known issue in the StudentLife app during the study
        sleep_hours = r.get('hour')
        if sleep_hours is None:
            sleep_hours = r.get('null')  # fall back to the buggy key

        all_sleep.append({
            'student_id':  student_id,
            'resp_time':   r.get('resp_time'),
            'sleep_hours': sleep_hours,
            'sleep_rate':  r.get('rate')
        })

sleep_raw = pd.DataFrame(all_sleep)
print(f'Raw sleep responses: {len(sleep_raw):,}')

In [ ]:
sleep_raw['datetime'] = convert_to_edt(sleep_raw['resp_time'])
sleep_raw = sleep_raw[within_study_period(sleep_raw['datetime'])].copy()

# Convert to numbers
sleep_raw['sleep_hours'] = pd.to_numeric(sleep_raw['sleep_hours'], errors='coerce')
sleep_raw['sleep_rate']  = pd.to_numeric(sleep_raw['sleep_rate'],  errors='coerce')

# Remove physiologically impossible values
# More than 16 hours or a negative number is clearly a data error
sleep_raw.loc[
    (sleep_raw['sleep_hours'] < 0) | (sleep_raw['sleep_hours'] > 16),
    'sleep_hours'
] = np.nan

# Quality rating must be 1-5
sleep_raw.loc[
    ~sleep_raw['sleep_rate'].between(1, 5) & sleep_raw['sleep_rate'].notna(),
    'sleep_rate'
] = np.nan

before = len(sleep_raw)
sleep_raw.drop_duplicates(subset=['student_id', 'resp_time'], keep='first', inplace=True)
print(f'Removed {before - len(sleep_raw)} duplicates')

# Add a sleep category column -- useful for visualisations
sleep_raw['sleep_category'] = pd.cut(
    sleep_raw['sleep_hours'],
    bins=[-1, 5, 7, 9, 99],
    labels=['Under 5h (deprived)', '5-7h (short)', '7-9h (adequate)', 'Over 9h (long)']
)

sleep_raw['date']       = sleep_raw['datetime'].dt.date
sleep_raw['study_week'] = get_study_week(sleep_raw['datetime'])

sleep_clean = sleep_raw[[
    'student_id', 'datetime', 'date', 'study_week',
    'sleep_hours', 'sleep_rate', 'sleep_category'
]].copy()

sleep_clean.sort_values(['student_id', 'datetime'], inplace=True)
sleep_clean.reset_index(drop=True, inplace=True)

print(f'\nFinal sleep dataset: {len(sleep_clean):,} rows from {sleep_clean["student_id"].nunique()} students')
print(f'Mean sleep hours:  {sleep_clean["sleep_hours"].mean():.2f}h')
print(f'Mean sleep quality: {sleep_clean["sleep_rate"].mean():.2f} / 5')
print(f'Nights under 7h:   {(sleep_clean["sleep_hours"] < 7).mean()*100:.1f}%')

## Step 8 -- Exercise EMA

The exercise survey records whether a student exercised on a given day, the intensity, and whether they walked.

**Fields:**
- `have` -- did they exercise today? (1 = yes, 2 = no)
- `exercise` -- exercise intensity (1-5)
- `walk` -- did they walk today? (1 = yes, 2 = no)
- `schedule` -- are they keeping to a fitness schedule? (1 = yes, 2 = no)

Note: `have` and `walk` use 1/2 coding rather than True/False -- I convert them to proper booleans.

In [ ]:
all_exercise = []

for student_id in ALL_STUDENTS:
    for r in read_ema_json('Exercise', student_id):
        all_exercise.append({
            'student_id': student_id,
            'resp_time':  r.get('resp_time'),
            'have':       r.get('have'),     # exercised? 1=yes 2=no
            'exercise':   r.get('exercise'), # intensity 1-5
            'walk':       r.get('walk'),     # walked? 1=yes 2=no
            'schedule':   r.get('schedule') # on schedule? 1=yes 2=no
        })

exercise_raw = pd.DataFrame(all_exercise)
print(f'Raw exercise responses: {len(exercise_raw):,}')

In [ ]:
exercise_raw['datetime'] = convert_to_edt(exercise_raw['resp_time'])
exercise_raw = exercise_raw[within_study_period(exercise_raw['datetime'])].copy()

# Convert 1/2 coding to True/False booleans
exercise_raw['exercised'] = pd.to_numeric(
    exercise_raw['have'], errors='coerce').map({1: True, 2: False})
exercise_raw['walked'] = pd.to_numeric(
    exercise_raw['walk'], errors='coerce').map({1: True, 2: False})
exercise_raw['on_schedule'] = pd.to_numeric(
    exercise_raw['schedule'].replace('null', np.nan), errors='coerce'
).map({1: True, 2: False})

# Intensity must be 1-5
exercise_raw['intensity'] = pd.to_numeric(exercise_raw['exercise'], errors='coerce')
exercise_raw.loc[
    ~exercise_raw['intensity'].between(1, 5) & exercise_raw['intensity'].notna(),
    'intensity'
] = np.nan

before = len(exercise_raw)
exercise_raw.drop_duplicates(subset=['student_id', 'resp_time'], keep='first', inplace=True)
print(f'Removed {before - len(exercise_raw)} duplicates')

exercise_raw['date']        = exercise_raw['datetime'].dt.date
exercise_raw['hour']        = exercise_raw['datetime'].dt.hour
exercise_raw['day_of_week'] = exercise_raw['datetime'].dt.dayofweek
exercise_raw['is_weekend']  = exercise_raw['day_of_week'] >= 5
exercise_raw['study_week']  = get_study_week(exercise_raw['datetime'])

exercise_clean = exercise_raw[[
    'student_id', 'datetime', 'date', 'hour',
    'day_of_week', 'is_weekend', 'study_week',
    'exercised', 'intensity', 'walked', 'on_schedule'
]].copy()

exercise_clean.sort_values(['student_id', 'datetime'], inplace=True)
exercise_clean.reset_index(drop=True, inplace=True)

exercise_rate = exercise_clean['exercised'].mean() * 100
print(f'\nFinal exercise dataset: {len(exercise_clean):,} rows from {exercise_clean["student_id"].nunique()} students')
print(f'Exercise rate: {exercise_rate:.1f}% of days')

## Step 9 -- Social EMA

The social survey asks how many people the student interacted with today.

**Field:** `number` -- count of people interacted with

**Known data bug:** Some responses store the count under the key `null` instead of `number`. But there is a further complication -- some of those `null` entries contain GPS coordinates (e.g. `43.759,-72.329`) instead of a person count. I detect these by checking for a comma in the string and skip them.

In [ ]:
all_social = []

for student_id in ALL_STUDENTS:
    for r in read_ema_json('Social', student_id):

        val = r.get('number')

        # BUG FIX: Some entries store the count under 'null' key
        # But some 'null' values are GPS coordinates -- skip those
        if val is None:
            raw_null = str(r.get('null', ''))
            if ',' not in raw_null and raw_null not in ['', 'null', 'None', 'Unknown']:
                val = raw_null  # looks like a valid count, not a GPS coordinate

        all_social.append({
            'student_id': student_id,
            'resp_time':  r.get('resp_time'),
            'social_n':   val
        })

social_raw = pd.DataFrame(all_social)
print(f'Raw social responses: {len(social_raw):,}')

In [ ]:
social_raw['datetime'] = convert_to_edt(social_raw['resp_time'])
social_raw = social_raw[within_study_period(social_raw['datetime'])].copy()

social_raw['social_n'] = pd.to_numeric(social_raw['social_n'], errors='coerce')

# Cap at 100 -- anything above that is likely a data entry error
social_raw.loc[social_raw['social_n'] > 100, 'social_n'] = np.nan

before = len(social_raw)
social_raw.drop_duplicates(subset=['student_id', 'resp_time'], keep='first', inplace=True)
print(f'Removed {before - len(social_raw)} duplicates')

social_raw['date']        = social_raw['datetime'].dt.date
social_raw['day_of_week'] = social_raw['datetime'].dt.dayofweek
social_raw['is_weekend']  = social_raw['day_of_week'] >= 5
social_raw['study_week']  = get_study_week(social_raw['datetime'])

social_clean = social_raw[[
    'student_id', 'datetime', 'date',
    'day_of_week', 'is_weekend', 'study_week', 'social_n'
]].copy()

social_clean.sort_values(['student_id', 'datetime'], inplace=True)
social_clean.reset_index(drop=True, inplace=True)

print(f'\nFinal social dataset: {len(social_clean):,} rows from {social_clean["student_id"].nunique()} students')
print(f'Mean people per day: {social_clean["social_n"].mean():.2f}')

## Step 10 -- Passive Sensing: Physical Activity

The activity sensor records the student's physical state every few seconds using the accelerometer. The values are: 0 = stationary, 1 = walking, 2 = running, 3 = unknown.

This dataset is enormous -- around 33 million rows across all students. I keep the raw readings here and aggregate to daily fractions in the master table step.

**Note on the column name:** The activity column in the CSV has a leading space in its name (`' activity'` not `'activity'`). This is a data quality issue I fix during loading using `str.strip()`.

In [ ]:
ACTIVITY_LABELS = {0: 'stationary', 1: 'walking', 2: 'running', 3: 'unknown'}

all_activity = []

for student_id in ALL_STUDENTS:
    df = read_sensing_csv('activity', student_id)
    if df is None:
        continue

    # FIX: Strip whitespace from column names to handle the leading-space bug
    df.columns = df.columns.str.strip()

    df['datetime'] = convert_to_edt(df['time'])
    df = df[within_study_period(df['datetime'])].copy()

    if len(df) == 0:
        continue

    df['activity_code']  = pd.to_numeric(df.get('activity', df.iloc[:, 1]), errors='coerce')
    df['activity_label'] = df['activity_code'].map(ACTIVITY_LABELS)
    df['student_id']     = student_id
    df['date']           = df['datetime'].dt.date
    df['hour']           = df['datetime'].dt.hour
    df['study_week']     = get_study_week(df['datetime'])

    all_activity.append(df[[
        'student_id', 'datetime', 'date', 'hour',
        'study_week', 'activity_code', 'activity_label'
    ]])

activity_clean = pd.concat(all_activity, ignore_index=True)

print(f'Activity dataset: {len(activity_clean):,} rows from {activity_clean["student_id"].nunique()} students')
print('Activity breakdown:')
print(activity_clean['activity_label'].value_counts())

## Step 11 -- Passive Sensing: Conversations

The conversation sensor uses the phone microphone to detect when a student is talking. It records the start and end time of each conversation event.

I calculate duration in minutes and apply two filters:
- Remove events under 30 seconds -- these are likely noise or mic activations, not real conversations
- Cap at 3 hours -- beyond that it is almost certainly a sensor glitch


In [ ]:
all_conv = []

for student_id in ALL_STUDENTS:
    df = read_sensing_csv('conversation', student_id)
    if df is None:
        continue

    df.columns = df.columns.str.strip()

    # Calculate conversation duration from start and end timestamps
    df['start_dt'] = convert_to_edt(df.get('start_timestamp', df.iloc[:, 0]))
    df['end_dt']   = convert_to_edt(df.get('end_timestamp',   df.iloc[:, 1]))

    df = df[within_study_period(df['start_dt'])].copy()
    if len(df) == 0:
        continue

    df['duration_min'] = (df['end_dt'] - df['start_dt']).dt.total_seconds() / 60

    # Remove noise: anything under 30 seconds is not a real conversation
    df = df[df['duration_min'] >= 0.5].copy()

    # Cap at 3 hours -- beyond that it is likely a sensor glitch
    df['duration_min'] = df['duration_min'].clip(upper=180)

    df['student_id'] = student_id
    df['date']       = df['start_dt'].dt.date
    df['hour']       = df['start_dt'].dt.hour
    df['study_week'] = get_study_week(df['start_dt'])

    all_conv.append(df[[
        'student_id', 'start_dt', 'end_dt', 'date',
        'hour', 'study_week', 'duration_min'
    ]])

conv_clean = pd.concat(all_conv, ignore_index=True)

print(f'Conversation dataset: {len(conv_clean):,} events from {conv_clean["student_id"].nunique()} students')
print(f'Mean duration: {conv_clean["duration_min"].mean():.1f} minutes')
print(f'Median duration: {conv_clean["duration_min"].median():.1f} minutes')

## Step 12 -- Passive Sensing: Bluetooth Proximity

The Bluetooth sensor periodically scans for nearby devices. The number of Bluetooth devices detected nearby is used as a proxy for social proximity -- the more devices, the more people are physically close to the student.

I keep the raw scan data here. The daily master table in the next step will compute daily summaries.

In [ ]:
all_bt = []

for student_id in ALL_STUDENTS:
    df = read_sensing_csv('bluetooth', student_id)
    if df is None:
        continue

    df.columns = df.columns.str.strip()

    # First column is the timestamp, second is the device count
    df['datetime']       = convert_to_edt(df.iloc[:, 0])
    df['devices_nearby'] = pd.to_numeric(df.iloc[:, 1], errors='coerce')

    df = df[within_study_period(df['datetime'])].copy()
    if len(df) == 0:
        continue

    df['student_id'] = student_id
    df['date']       = df['datetime'].dt.date
    df['hour']       = df['datetime'].dt.hour
    df['study_week'] = get_study_week(df['datetime'])

    all_bt.append(df[[
        'student_id', 'datetime', 'date', 'hour', 'study_week', 'devices_nearby'
    ]])

bt_clean = pd.concat(all_bt, ignore_index=True)

print(f'Bluetooth dataset: {len(bt_clean):,} scans from {bt_clean["student_id"].nunique()} students')
print(f'Mean devices nearby: {bt_clean["devices_nearby"].mean():.2f}')

## Step 13 -- Build the Daily Master Table

This is the most important output of the cleaning stage. I aggregate everything into a single table with **one row per student per day**.

This master table becomes the primary input for both the EDA and machine learning stages. Each row represents everything known about one student on one day -- their reported stress, sleep, mood, exercise, and what the sensors recorded.

In [ ]:
# Daily stress -- mean level and response count per day
daily_stress = (
    stress_clean
    .groupby(['student_id', 'date', 'study_week'])
    .agg(
        stress_avg   = ('stress_level', 'mean'),
        stress_count = ('stress_level', 'count')
    ).reset_index()
)

# Daily mood -- mean mood score
daily_mood = (
    mood_clean
    .groupby(['student_id', 'date', 'study_week'])
    .agg(mood_score_avg = ('mood_score', 'mean'))
    .reset_index()
)

# Daily sleep -- use the most recent reading of the day
daily_sleep = (
    sleep_clean
    .sort_values('datetime')
    .groupby(['student_id', 'date', 'study_week'])
    .agg(
        sleep_hours = ('sleep_hours', 'last'),
        sleep_rate  = ('sleep_rate',  'last')
    ).reset_index()
)

# Daily exercise -- True if they exercised at any point during the day
daily_exercise = (
    exercise_clean
    .groupby(['student_id', 'date', 'study_week'])
    .agg(
        exercised_today = ('exercised', 'max'),
        walked_today    = ('walked',    'max')
    ).reset_index()
)

# Daily social -- mean number of people interacted with
daily_social = (
    social_clean
    .groupby(['student_id', 'date', 'study_week'])
    .agg(social_n_avg = ('social_n', 'mean'))
    .reset_index()
)

# Daily conversation -- total talking time in minutes
daily_conv = (
    conv_clean
    .groupby(['student_id', 'date', 'study_week'])
    .agg(
        total_talking_minutes = ('duration_min', 'sum'),
        num_conversations     = ('duration_min', 'count')
    ).reset_index()
)

# Daily Bluetooth -- average and unique device count
daily_bt = (
    bt_clean
    .groupby(['student_id', 'date', 'study_week'])
    .agg(
        avg_devices_nearby    = ('devices_nearby', 'mean'),
        unique_devices_nearby = ('devices_nearby', 'nunique')
    ).reset_index()
)

# Daily activity -- fraction of sensor readings in each physical state
daily_activity = (
    activity_clean
    .groupby(['student_id', 'date', 'study_week'])
    .apply(lambda x: pd.Series({
        'fraction_stationary': (x['activity_code'] == 0).mean(),
        'fraction_walking':    (x['activity_code'] == 1).mean(),
        'fraction_running':    (x['activity_code'] == 2).mean(),
    })).reset_index()
)

print('All daily aggregations complete.')

In [ ]:
from functools import reduce

# Merge all daily tables together on student_id and date
# Using outer joins so that a student-day appears even if they only
# completed some surveys that day
all_daily_tables = [
    daily_stress, daily_mood, daily_sleep, daily_exercise,
    daily_social, daily_conv, daily_bt, daily_activity
]

master = reduce(
    lambda left, right: pd.merge(
        left, right,
        on=['student_id', 'date', 'study_week'],
        how='outer'
    ),
    all_daily_tables
)

# Add useful calendar columns
master['date']        = pd.to_datetime(master['date'])
master['day_of_week'] = master['date'].dt.dayofweek
master['is_weekend']  = master['day_of_week'] >= 5

master.sort_values(['student_id', 'date'], inplace=True)
master.reset_index(drop=True, inplace=True)

print(f'Daily master table: {len(master):,} rows')
print(f'Students:  {master["student_id"].nunique()}')
print(f'Date range: {master["date"].min().date()} to {master["date"].max().date()}')
print(f'Columns:   {list(master.columns)}')
master.head(3)

## Step 14 -- Sanity Checks

Before saving anything, I run three quick visual checks:

1. Stress by week should show an upward trend toward finals (week 10)
2. Sleep should average roughly 7-9 hours
3. The number of survey responses should be highest early in the term

These charts also give me a first look at the data to confirm nothing looks wrong.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Chart 1: Mean stress level by study week
week_stress = stress_clean.groupby('study_week')['stress_level'].mean()
axes[0].bar(week_stress.index, week_stress.values, color='#E05C5C')
axes[0].axhline(
    y=stress_clean['stress_level'].mean(),
    color='black', linestyle='--', linewidth=1, label='Overall mean'
)
axes[0].set_xlabel('Study Week')
axes[0].set_ylabel('Mean Stress Level')
axes[0].set_title('Stress Level by Week')
axes[0].legend()
axes[0].set_ylim(0, 5)

# Chart 2: Sleep hours distribution
axes[1].hist(sleep_clean['sleep_hours'].dropna(), bins=20,
             color='#4ECDC4', edgecolor='white')
axes[1].axvline(x=7, color='red', linestyle='--', label='7h minimum')
axes[1].set_xlabel('Hours of Sleep')
axes[1].set_ylabel('Number of Nights')
axes[1].set_title('Sleep Hours Distribution')
axes[1].legend()

# Chart 3: Survey response count by week
week_responses = stress_clean.groupby('study_week').size()
axes[2].bar(week_responses.index, week_responses.values, color='#7C6AF7')
axes[2].set_xlabel('Study Week')
axes[2].set_ylabel('Number of Responses')
axes[2].set_title('Stress Responses per Week')

plt.suptitle('Sanity Checks -- Data Cleaning Output', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cleaning_sanity_checks.png', dpi=120, bbox_inches='tight')
plt.show()
print('Charts saved. Cleaning output looks reasonable.')

## Step 15 -- Save All Clean Files to Google Drive

I save all 9 cleaned datasets plus the daily master table as CSV files to a folder called `clean_data` on Drive. These files are the inputs for the EDA and ML notebooks.

In [ ]:
SAVE_DIR = '/content/drive/MyDrive/clean_data'
os.makedirs(SAVE_DIR, exist_ok=True)

files_to_save = {
    'mood_clean.csv':         mood_clean,
    'stress_clean.csv':       stress_clean,
    'sleep_clean.csv':        sleep_clean,
    'exercise_clean.csv':     exercise_clean,
    'social_clean.csv':       social_clean,
    'conversation_clean.csv': conv_clean,
    'activity_clean.csv':     activity_clean,
    'bluetooth_clean.csv':    bt_clean,
    'daily_master.csv':       master,
}

print('Saving files to Google Drive...')
print()

for filename, dataframe in files_to_save.items():
    filepath = os.path.join(SAVE_DIR, filename)
    dataframe.to_csv(filepath, index=False)
    size_kb = os.path.getsize(filepath) / 1024
    print(f'  {filename:<30} {len(dataframe):>10,} rows   {size_kb:>8.0f} KB')

print()
print(f'All 9 files saved successfully to: {SAVE_DIR}')

## Summary

The data cleaning stage is now complete. Here is a summary of what was done:

| Step | What I did | Key decisions |
|------|-----------|---------------|
| 1-2 | Mounted Drive, imported libraries | -- |
| 3 | Set file paths | `DATA_DIR` points to the raw dataset folder |
| 4 | Defined helper functions | Convert timestamps to EDT (critical); study week mapping |
| 5 | Cleaned Mood EMA | Three survey folders merged; mood score = happy minus sad |
| 6 | Cleaned Stress EMA | Main outcome variable; validated range 1-5 |
| 7 | Cleaned Sleep EMA | Fixed `null` key bug; removed impossible values |
| 8 | Cleaned Exercise EMA | Converted 1/2 coding to True/False booleans |
| 9 | Cleaned Social EMA | Fixed `null` key bug; removed GPS coordinates |
| 10 | Cleaned Activity sensing | Fixed leading-space column name; 33M+ rows |
| 11 | Cleaned Conversation sensing | Removed events under 30s; capped at 3 hours |
| 12 | Cleaned Bluetooth sensing | Raw proximity scans kept |
| 13 | Built daily master table | One row per student per day; all variables merged |
| 14 | Sanity checks | Visual confirmation patterns look reasonable |
| 15 | Saved to Drive | 9 CSV files + daily master ready for EDA |

**Output files saved to `clean_data/` on Google Drive:**
- `mood_clean.csv`, `stress_clean.csv`, `sleep_clean.csv`
- `exercise_clean.csv`, `social_clean.csv`
- `conversation_clean.csv`, `activity_clean.csv`, `bluetooth_clean.csv`
- `daily_master.csv` -- primary input for EDA and ML notebooks
